# AIC 2026 — Visual embedding ingestion · SigLIP 2 So400m

**Input:** `aqpahm/aic2026-keyframes-transnetv2`  
**Output:** `aqpahm/aic2026-visual-siglip2-so400m`  
**Model:** `google/siglip2-so400m-patch16-384`

Notebook nguồn chạy độc lập trên **Google Colab hoặc Kaggle**. Embedding được ingest
độc lập; notebook này không quyết định cách index, fusion hay kiến trúc retrieval.


In [ ]:
%pip install -q -U "huggingface_hub>=0.34,<2" "transformers==4.57.1" "safetensors>=0.4" "pyarrow>=16" "accelerate>=1.0"


In [ ]:
import os
import sys
import tempfile

# Set these before importing huggingface_hub.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import hashlib
import io
import json
import math
import re
import shutil
import tarfile
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from safetensors import safe_open
from safetensors.torch import save_file
from transformers import AutoModel, AutoProcessor

ImageFile.LOAD_TRUNCATED_IMAGES = True

INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
OUTPUT_REPO = "aqpahm/aic2026-visual-siglip2-so400m"
MODEL_ID = "google/siglip2-so400m-patch16-384"

# A T4 normally handles 16 images at 384 px. OOM recovery halves this value.
INITIAL_BATCH_SIZE = 16
UPLOAD_BATCH_VIDEOS = 5
MAX_VIDEOS_PER_RUN = 5  # pilot; set to None after validating the first run
OUTPUT_PRIVATE = True
EXPECTED_DIMENSION = 1152

def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata
        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(
            f"Missing {name}. Add it as a Colab/Kaggle secret and enable notebook access."
        )
    return value


def hosted_work_root(job_name):
    if RUNTIME == "colab":
        return Path("/content") / job_name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / job_name
    return Path(tempfile.gettempdir()) / job_name

RUNTIME = detect_runtime()
ROOT = hosted_work_root("aic-visual-siglip2")
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

HF_TOKEN = read_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME)
print("Hugging Face:", account["name"])

api.create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=OUTPUT_PRIVATE,
    exist_ok=True,
)
INPUT_REVISION = api.dataset_info(INPUT_REPO, token=HF_TOKEN).sha
MODEL_REVISION = api.model_info(MODEL_ID, token=HF_TOKEN).sha
print("Input revision:", INPUT_REVISION)
print("Model revision:", MODEL_REVISION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in the Colab/Kaggle runtime before running this job")

DEVICE = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
torch.backends.cudnn.benchmark = True



In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(DEVICE).eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

print("Model loaded:", MODEL_ID)
print("Model dtype:", next(model.parameters()).dtype)



In [ ]:
def retry(operation, description, attempts=7):
    """Retry network operations with capped exponential backoff."""
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(300, 5 * (2 ** (attempt - 1)))
            print(
                f"{description} failed ({type(error).__name__}); "
                f"retry in {delay}s [{attempt}/{attempts}]"
            )
            time.sleep(delay)


def download_hf_file(filename, local_dir):
    """Download a pinned input file and reuse partial/cache data on retry."""
    local_dir.mkdir(parents=True, exist_ok=True)
    return Path(
        retry(
            lambda: hf_hub_download(
                repo_id=INPUT_REPO,
                repo_type="dataset",
                filename=filename,
                revision=INPUT_REVISION,
                token=HF_TOKEN,
                local_dir=local_dir,
            ),
            f"download {filename}",
        )
    )


def normalized_tar_name(value):
    return PurePosixPath(str(value).replace("\\", "/")).as_posix().lstrip("./")


def build_tar_lookup(archive):
    by_name = {}
    by_basename = defaultdict(list)
    for member in archive.getmembers():
        if not member.isfile():
            continue
        name = normalized_tar_name(member.name)
        by_name[name] = member
        by_basename[PurePosixPath(name).name].append(member)
    return by_name, by_basename


def locate_tar_member(image_path, by_name, by_basename):
    expected = normalized_tar_name(image_path)
    if expected in by_name:
        return by_name[expected]
    basename = PurePosixPath(expected).name
    matches = by_basename.get(basename, [])
    if len(matches) != 1:
        raise RuntimeError(
            f"Cannot uniquely locate {image_path!r} inside keyframes.tar"
        )
    return matches[0]


def read_image_batch(archive, records, by_name, by_basename):
    images = []
    for record in records:
        member = locate_tar_member(record["image_path"], by_name, by_basename)
        stream = archive.extractfile(member)
        if stream is None:
            raise RuntimeError(f"Cannot read tar member {member.name}")
        payload = stream.read()
        with Image.open(io.BytesIO(payload)) as source:
            image = source.convert("RGB")
            image.load()
        images.append(image)
    return images


def image_features(images):
    inputs = processor(images=images, return_tensors="pt")
    kwargs = {}
    for name in ("pixel_values", "pixel_attention_mask", "spatial_shapes"):
        if name not in inputs:
            continue
        tensor = inputs[name]
        if torch.is_floating_point(tensor):
            tensor = tensor.to(dtype=torch.float16)
        kwargs[name] = tensor.to(DEVICE, non_blocking=True)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        features = model.get_image_features(**kwargs)
    if not torch.is_tensor(features):
        raise RuntimeError(f"Unexpected feature type: {type(features).__name__}")
    # Normalize in FP32, then store compact FP16 vectors.
    features = torch.nn.functional.normalize(features.float(), dim=-1)
    return features.half().cpu()


def embed_archive(archive, records, initial_batch_size):
    """Embed aligned tar members; reduce batch size after CUDA OOM."""
    by_name, by_basename = build_tar_lookup(archive)
    chunks = []
    cursor = 0
    active_batch = min(initial_batch_size, len(records))
    minimum_batch = active_batch

    while cursor < len(records):
        selected = records[cursor : cursor + active_batch]
        images = read_image_batch(archive, selected, by_name, by_basename)
        try:
            chunk = image_features(images)
        except (torch.OutOfMemoryError, RuntimeError) as error:
            is_oom = isinstance(error, torch.OutOfMemoryError) or "out of memory" in str(error).lower()
            if not is_oom or active_batch == 1:
                raise
            active_batch = max(1, active_batch // 2)
            minimum_batch = min(minimum_batch, active_batch)
            print(f"CUDA OOM; reducing batch size to {active_batch}")
            gc.collect()
            torch.cuda.empty_cache()
            continue
        finally:
            for image in images:
                image.close()

        if chunk.ndim != 2 or chunk.shape[1] != EXPECTED_DIMENSION:
            raise RuntimeError(
                f"Expected [N, {EXPECTED_DIMENSION}] embeddings, got {tuple(chunk.shape)}"
            )
        chunks.append(chunk)
        cursor += len(selected)
        if cursor % 256 < len(selected) or cursor == len(records):
            print(f"  embedded {cursor}/{len(records)}")

    embeddings = torch.cat(chunks, dim=0)
    if len(embeddings) != len(records):
        raise RuntimeError(f"Embedding count mismatch: {len(embeddings)} vs {len(records)}")
    if not torch.isfinite(embeddings).all():
        raise RuntimeError("Embedding tensor contains NaN or infinity")
    norms = torch.linalg.vector_norm(embeddings.float(), dim=1)
    max_norm_error = float((norms - 1).abs().max())
    if max_norm_error > 0.005:
        raise RuntimeError(f"L2 normalization check failed: max error {max_norm_error}")
    return embeddings.contiguous(), minimum_batch, max_norm_error



In [ ]:
IDENTITY_COLUMNS = [
    "video_id",
    "frame_uid",
    "sample_n",
    "frame_idx",
    "shot_id",
    "timestamp_sec",
    "image_path",
    "embedding_row",
]


def make_manifest(metadata, video_id):
    required = {"video_id", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"}
    missing = required - set(metadata.columns)
    if missing:
        raise RuntimeError(f"{video_id}: source columns missing: {sorted(missing)}")
    if metadata.empty:
        raise RuntimeError(f"{video_id}: source metadata is empty")

    manifest = metadata.sort_values("sample_n", kind="stable").reset_index(drop=True).copy()
    if manifest["video_id"].astype(str).nunique() != 1:
        raise RuntimeError(f"{video_id}: metadata contains multiple video IDs")
    if str(manifest.iloc[0]["video_id"]) != video_id:
        raise RuntimeError(f"{video_id}: source video ID does not match path")
    if manifest["sample_n"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate sample_n values")
    if manifest["frame_idx"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate frame_idx values")

    manifest["video_id"] = manifest["video_id"].astype(str)
    manifest["sample_n"] = manifest["sample_n"].astype("int64")
    manifest["frame_idx"] = manifest["frame_idx"].astype("int64")
    manifest["shot_id"] = manifest["shot_id"].astype("int64")
    manifest["timestamp_sec"] = manifest["timestamp_sec"].astype("float64")
    manifest["image_path"] = manifest["image_path"].astype(str)
    manifest["frame_uid"] = [
        f"{video_id}:{frame_idx}"
        for frame_idx in manifest["frame_idx"].tolist()
    ]
    manifest["embedding_row"] = np.arange(len(manifest), dtype=np.int64)
    return manifest[IDENTITY_COLUMNS]


def catalog_hash(manifest):
    hasher = hashlib.sha256()
    for row in manifest.itertuples(index=False):
        payload = {
            "video_id": row.video_id,
            "frame_uid": row.frame_uid,
            "sample_n": int(row.sample_n),
            "frame_idx": int(row.frame_idx),
            "shot_id": int(row.shot_id),
            "timestamp_sec": float(row.timestamp_sec),
            "image_path": row.image_path,
            "embedding_row": int(row.embedding_row),
        }
        hasher.update(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8"))
        hasher.update(b"\n")
    return hasher.hexdigest()


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    hasher = hashlib.sha256()
    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


def write_video_output(spec, manifest, embeddings, stats, max_norm_error):
    directory = OUTPUT_ROOT / spec["level"] / spec["video_id"]
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)

    embedding_path = directory / "embeddings.safetensors"
    frames_path = directory / "frames.parquet"
    marker_path = directory / "_VISUAL_SUCCESS.json"

    save_file(
        {"embeddings": embeddings},
        str(embedding_path),
        metadata={
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "normalized": "l2",
            "dtype": "float16",
            "schema_version": "1",
        },
    )
    manifest.to_parquet(frames_path, index=False)

    # Read the stored tensor back before publishing the success marker.
    with safe_open(str(embedding_path), framework="pt", device="cpu") as file:
        stored = file.get_tensor("embeddings")
    if tuple(stored.shape) != (len(manifest), EXPECTED_DIMENSION):
        raise RuntimeError(f"Stored tensor has invalid shape: {tuple(stored.shape)}")
    if stored.dtype != torch.float16:
        raise RuntimeError(f"Stored tensor has invalid dtype: {stored.dtype}")

    marker = {
        "video_id": spec["video_id"],
        "status": "success",
        "frames": len(manifest),
        "embedding_dimension": EXPECTED_DIMENSION,
        "embedding_dtype": "float16",
        "normalization": "l2",
        "max_norm_error_after_fp16": max_norm_error,
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "source_tar": spec["tar_filename"],
        "source_frames": spec["frames_filename"],
        "catalog_sha256": catalog_hash(manifest),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "processor_id": MODEL_ID,
        "image_size": 384,
        "embeddings_sha256": sha256_file(embedding_path),
        "stats": stats,
        "schema_version": 1,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [embedding_path, frames_path, marker_path]


def upload_outputs(paths, video_ids):
    operations = [
        CommitOperationAdd(
            path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
            path_or_fileobj=str(path),
        )
        for path in paths
    ]
    retry(
        lambda: api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add SigLIP 2 embeddings for {video_ids[0]} through {video_ids[-1]}",
            token=HF_TOKEN,
        ),
        f"upload {len(video_ids)} videos",
    )



In [ ]:
input_files = set(
    retry(
        lambda: api.list_repo_files(INPUT_REPO, repo_type="dataset", revision=INPUT_REVISION),
        "list input files",
    )
)
output_files = set(
    retry(
        lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
        "list output files",
    )
)

tar_pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = tar_pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    frames_filename = f"data/{level}/{video_id}/frames.parquet"
    if frames_filename not in input_files:
        raise RuntimeError(f"Missing source metadata: {frames_filename}")
    video_specs.append(
        {
            "level": level,
            "video_id": video_id,
            "tar_filename": filename,
            "frames_filename": frames_filename,
        }
    )


def remote_complete(spec, files):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    return all(
        f"{prefix}/{name}" in files
        for name in ("embeddings.safetensors", "frames.parquet", "_VISUAL_SUCCESS.json")
    )


completed_specs = [spec for spec in video_specs if remote_complete(spec, output_files)]
pending_specs = [spec for spec in video_specs if not remote_complete(spec, output_files)]
if MAX_VIDEOS_PER_RUN is not None:
    pending_specs = pending_specs[:MAX_VIDEOS_PER_RUN]

print("Source videos:", len(video_specs))
print("Already complete:", len(completed_specs))
print("Selected this run:", len(pending_specs))
if not video_specs:
    raise RuntimeError(
        f"No keyframe TAR files found in {INPUT_REPO}. Check INPUT_REPO after renaming."
    )



In [ ]:
def process_video(spec):
    video_id = spec["video_id"]
    task_dir = DOWNLOAD_ROOT / video_id
    shutil.rmtree(task_dir, ignore_errors=True)
    task_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(0)
    try:
        frames_path = download_hf_file(spec["frames_filename"], task_dir)
        tar_path = download_hf_file(spec["tar_filename"], task_dir)
        manifest = make_manifest(pd.read_parquet(frames_path), video_id)
        records = manifest.to_dict("records")

        with tarfile.open(tar_path, mode="r:*") as archive:
            embeddings, minimum_batch, max_norm_error = embed_archive(
                archive, records, INITIAL_BATCH_SIZE
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        stats = {
            "elapsed_sec": round(elapsed, 3),
            "frames_per_sec": round(len(manifest) / max(elapsed, 1e-6), 3),
            "peak_vram_gib": round(torch.cuda.max_memory_allocated(0) / 1024**3, 3),
            "minimum_batch_size": minimum_batch,
            "device": "cuda:0",
            "gpu": torch.cuda.get_device_name(0),
        }
        paths = write_video_output(
            spec, manifest, embeddings, stats, max_norm_error
        )
        return {
            "video_id": video_id,
            "paths": paths,
            "frames": len(manifest),
            "stats": stats,
        }
    finally:
        shutil.rmtree(task_dir, ignore_errors=True)
        gc.collect()
        torch.cuda.empty_cache()


pending_paths = []
pending_ids = []
run_results = []
failures = []


def flush_pending():
    global pending_paths, pending_ids
    if not pending_paths:
        return
    upload_outputs(pending_paths, pending_ids)
    print(f"Uploaded {len(pending_ids)} videos in one commit")
    for video_id in pending_ids:
        level = video_id.split("_")[0]
        shutil.rmtree(OUTPUT_ROOT / level / video_id, ignore_errors=True)
    pending_paths = []
    pending_ids = []


for position, spec in enumerate(pending_specs, 1):
    try:
        result = process_video(spec)
        run_results.append(result)
        pending_paths.extend(result["paths"])
        pending_ids.append(result["video_id"])
        stats = result["stats"]
        print(
            f"[{position}/{len(pending_specs)}] OK {result['video_id']}: "
            f"{result['frames']} frames, {stats['frames_per_sec']} FPS, "
            f"{stats['peak_vram_gib']} GiB"
        )
        if len(pending_ids) >= UPLOAD_BATCH_VIDEOS:
            flush_pending()
    except Exception as error:
        failures.append({"video_id": spec["video_id"], "error": repr(error)})
        print(f"[{position}/{len(pending_specs)}] FAILED {spec['video_id']}: {error!r}")

flush_pending()

print("=" * 72)
print("Processed this run:", len(run_results))
print("Failures:", len(failures))
if run_results:
    total_frames = sum(result["frames"] for result in run_results)
    total_seconds = sum(result["stats"]["elapsed_sec"] for result in run_results)
    print("Frames:", total_frames)
    print("End-to-end FPS:", round(total_frames / max(total_seconds, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError(
        "SigLIP 2 run has failures. Rerun after diagnosis; remote completed videos are skipped."
    )



In [ ]:
# Final remote audit. A pilot intentionally reports the remaining videos.
final_files = set(
    retry(
        lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
        "refresh output files",
    )
)
missing_videos = [
    spec["video_id"] for spec in video_specs if not remote_complete(spec, final_files)
]

print(f"Remote complete: {len(video_specs) - len(missing_videos)}/{len(video_specs)} videos")
print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
if missing_videos:
    print("Remaining:", len(missing_videos), "sample:", missing_videos[:20])
if MAX_VIDEOS_PER_RUN is None and missing_videos:
    raise RuntimeError(f"Missing visual outputs for {len(missing_videos)} videos")

if MAX_VIDEOS_PER_RUN is not None:
    print(
        "Pilot finished. Inspect the output, then set MAX_VIDEOS_PER_RUN = None "
        "and rerun all cells. Completed videos will be skipped."
    )
